In [ ]:
# (setup cell already installs what this notebook needs)

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

## Exercises: 5 rules of effective prompt engineering
5 rules of effective prompt engineering:
1. Clear instructions improve accuracy.
2. Examples stabilize the response.
3. A defined response format = a predictable result.
4. Breaking into steps = better and more complete solutions.
5. Testing and verification = safety and correctness.

In [2]:
# Import libraries
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI

# base model
llm = make_llm(temperature=0.7)

Exercise 1 \
Rule 1 - Clear instructions \
 Write your own example of a bad and a good prompt. List 2 differences in the results you get.

In [3]:
# Bad prompt - vague, no role or expectations
bad_prompt = "Write a function in Python."
print("=== Bad prompt ===")
print(llm.invoke(bad_prompt).content)

# Good prompt - clearly defined role and expectations
good_prompt = """You are an expert Python programmer.
Write a function in Python that takes a list of integers
and returns a new list containing only the even numbers.
Add a unit test in pytest."""
print("\n=== Good prompt ===")
print(llm.invoke(good_prompt).content)

=== Zły prompt ===
Oczywiście! Jaką funkcję chciałbyś, abym napisał? Możesz podać szczegóły dotyczące jej działania lub celu, a ja postaram się pomóc.

=== Dobry prompt ===
Oto przykładowa funkcja w Pythonie, która przyjmuje listę liczb całkowitych i zwraca nową listę zawierającą tylko liczby parzyste. Dodatkowo dodam test jednostkowy z użyciem biblioteki `pytest`.

### Funkcja

```python
def get_even_numbers(numbers):
    """Funkcja zwraca listę liczb parzystych z podanej listy."""
    return [num for num in numbers if num % 2 == 0]
```

### Test jednostkowy

Aby przetestować funkcję, stworzymy plik testowy, na przykład `test_even_numbers.py`.

```python
import pytest
from your_module import get_even_numbers  # Upewnij się, że importujesz funkcję z odpowiedniego modułu

def test_get_even_numbers():
    assert get_even_numbers([1, 2, 3, 4, 5, 6]) == [2, 4, 6]
    assert get_even_numbers([10, 11, 12, 13, 14]) == [10, 12, 14]
    assert get_even_numbers([-2, -1, 0, 1, 2]) == [-2, 0, 2]
 

Exercise 2 \
Rule 2 - Use examples \
Generate tags based on the company's website content. Gradually add a few more company-description examples and verify how the generated results change as the number of examples increases.


In [4]:
# Zero-shot
zero_shot = """Give tags describing the company based on the website text:
The company Lego produces toys for children."""
print("=== Zero-shot ===")
print(llm.invoke(zero_shot).content)

# One-shot
one_shot = """Give at most three tags describing the company based on the website text.
Example:
Text: The company Lego produces bricks for children.
Tags: toys, bricks, children

Now:
Text: The company Nike produces sportswear and sports shoes.
Tags:"""
print("\n=== One-shot ===")
print(llm.invoke(one_shot).content)

# Few-shot
few_shot = """Give at most three tags describing the company based on the website text.

Example 1:
Text: The company Lego produces bricks for children.
Tags: toys, bricks, children

Example 2:
Text: The company Nike produces sportswear and sports shoes.
Tags: sport, clothing, footwear

Now:
Text: The company Tesla produces electric cars and energy storage.
Tags:"""
print("\n=== Few-shot ===")
print(llm.invoke(few_shot).content)


=== Zero-shot ===
Oto kilka tagów, które mogą opisać firmę Lego na podstawie podanego tekstu:

1. LEGO
2. zabawki
3. dzieci
4. producent zabawek
5. kreatywność
6. konstrukcyjne
7. edukacyjne
8. rozrywka
9. rozwój dzieci
10. marka globalna

=== One-shot ===
odzież, buty, sport

=== Few-shot ===
samochody, elektryczność, energia


Exercise 3 \
Rule 3 - Define the response format \
Analyze and modify the example below so that it concerns generating a recipe for your favorite dish instead of a trip plan. \
Add input and output validation rules using the Pydantic library.

In [5]:
from pydantic import BaseModel, Field
from typing import List
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import dotenv

dotenv.load_dotenv()

# Output schema definition (Pydantic)
class CityGuide(BaseModel):
    city: str = Field(..., description="The city the guide is about")
    summary: str = Field(..., description="A short description of the city (2–3 sentences)")
    must_do: List[str] = Field(..., description="A list of 3–5 things to do")

# LLM + structured output
llm = make_llm()
structured_llm = llm.with_structured_output(CityGuide)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a travel expert. Answer in Polish, concisely."),
    ("user", "Create a short guide to {city} for a {days}-day visit.")
])

chain = prompt | structured_llm

# Pydantic input validation + chain invocation
from pydantic import BaseModel, Field

class GuideRequest(BaseModel):
    city: str = Field(min_length=2)
    days: int = Field(ge=1, le=7)

req = GuideRequest(city="Poznań", days=2)

result: CityGuide = chain.invoke(req.model_dump())  # <- you get a Pydantic OBJECT
print(result)                         # CityGuide(city=..., summary=..., must_do=[...])
print(result.model_dump_json(indent=2))  # JSON ready to save/transport

city='Poznań' summary='Poznań to jedno z najstarszych miast w Polsce, znane z bogatej historii, pięknej architektury i tętniącej życiem atmosfery. To idealne miejsce na krótki wypad, oferujące zarówno zabytki, jak i nowoczesne atrakcje.' must_do=['Odwiedź Stary Rynek i zobacz ratusz z koziołkami.', 'Spaceruj po Ostrówie Tumskim, historycznym centrum Poznania.', 'Zrelaksuj się w Parku Cytadela, gdzie znajdziesz muzea i piękne tereny zielone.', 'Spróbuj lokalnych specjałów w jednej z restauracji, np. pyry z gzikiem.', 'Zobacz Muzeum Narodowe z bogatą kolekcją sztuki.']
{
  "city": "Poznań",
  "summary": "Poznań to jedno z najstarszych miast w Polsce, znane z bogatej historii, pięknej architektury i tętniącej życiem atmosfery. To idealne miejsce na krótki wypad, oferujące zarówno zabytki, jak i nowoczesne atrakcje.",
  "must_do": [
    "Odwiedź Stary Rynek i zobacz ratusz z koziołkami.",
    "Spaceruj po Ostrówie Tumskim, historycznym centrum Poznania.",
    "Zrelaksuj się w Parku Cytadel

Exercise 4 \
Rule 4 - Break complex tasks into steps \
Modify the example below so that it accomplishes the task in several steps: "Advise me on what car I should buy?".

In [6]:
# Bad prompt - everything at once
bad_prompt = """Prepare a three-day sightseeing plan for Poznań with a budget of 300 euros,
including attractions, restaurants, transport and maps."""
print("=== Bad prompt ===")
print(llm.invoke(bad_prompt).content[:600], "...")

# Good prompt - step by step
good_step1 = "List the most important cultural attractions in Poznań with their opening hours."
step1 = llm.invoke(good_step1).content
print("\n=== Good prompt - step 1 ===")
print(step1[:600], "...")

good_step2 = f"Based on this list, create a 3-day sightseeing plan, max 4 attractions per day. Attractions: {step1}"
step2 = llm.invoke(good_step2).content
print("\n=== Good prompt - step 2 ===")
print(step2[:600], "...")


=== Zły prompt ===
Oto trzydniowy plan zwiedzania Poznania z budżetem 300 euro. Plan uwzględnia atrakcje turystyczne, restauracje, transport oraz mapy.

### Dzień 1: Stare Miasto i okolice

**Rano:**
- **Śniadanie:** Kawiarnia "Café La Ruina" (około 5 euro)
- **Atrakcja:** Stary Rynek, Ratusz (bezpłatnie)
- **Transport:** Spacer po Starym Mieście (bezpłatnie)

**Południe:**
- **Atrakcja:** Muzeum Narodowe (bilet wstępu około 5 euro)
- **Obiad:** Restauracja "Bamberka" (około 10 euro)

**Popołudnie:**
- **Atrakcja:** Ostrów Tumski, Katedra (bezpłatnie, ewentualnie 2 euro za wejście do katedry)
- **Transport:** Sp ...

=== Dobry prompt — krok 1 ===
Oto lista najważniejszych atrakcji kulturalnych w Poznaniu wraz z ich godzinami otwarcia. Proszę pamiętać, że godziny otwarcia mogą się zmieniać, dlatego zawsze warto sprawdzić aktualne informacje na oficjalnych stronach internetowych.

1. **Stary Rynek**
   - Opis: Serce Poznania, z pięknymi kamienicami i ratuszem.
   - Godziny otwarcia: Cało

Exercise 5 \
Rule 5 - Test and verify the results
1. Define a scoring range (scale 1–5) - write a dedicated function in Python (e.g. 0.0 - 0.2 -> 1, 0.21 - 0.4 -> 2, etc.).
2. Score the prepared response on the prepared scale.

In [7]:
from langchain_classic.evaluation import load_evaluator
from dotenv import load_dotenv

load_dotenv()

evaluator = load_evaluator("embedding_distance", embeddings_model="openai")

result = evaluator.evaluate_strings(
    prediction="The capital of Poland is Warsaw",
    reference="Warsaw is the capital of Poland"
)

print(result)


{'score': 0.055613485077632974}
